# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PrishaSolanki-coder/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

## Finding 1: Predictive performance

The FlyRank research paper reports that machine-learning models can help identify content that may need attention. The label is based on observed content performance and decline behavior. A methodology question is whether the validation setup fully represents how the model would perform on genuinely unseen clients or future data. This matters because random splits can place very similar pages from the same client in both training and validation.

## Finding 2: Content refresh signals

The paper uses measurable content and search-performance signals to identify pages that may be candidates for refresh. The label comes from observed changes in content performance rather than from a human judgment about whether a page "needs" a refresh. A methodology question is whether the available historical window and validation design are sufficient to support the broader claim that these signals generalize to other clients and time periods.

These questions do not invalidate the findings. They identify where additional grouped or time-aware validation would make the research claim stronger.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Clients:", df["client_id"].nunique())
print("Declining pages:", df["is_declining_label"].sum())
print(
    "Declining rate:",
    round(df["is_declining_label"].mean() * 100, 2),
    "%"
)


Rows: 30000
Columns: 45
Clients: 32
Declining pages: 16262
Declining rate: 54.21 %


## 2. My model under an honest split (before/after)

## Validation comparison

I will compare the Week-5 validation result with a stricter client-grouped validation result. The grouped split keeps all pages from a client in either training or validation, so the model cannot learn from other pages belonging to the same client. I use Precision@50 because the capstone is a ranking problem where the content team would review a small number of high-priority pages.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression


# -----------------------------
# Load data
# -----------------------------

url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)


# -----------------------------
# Feature set
# -----------------------------

numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

categorical_features = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used"
]

feature_columns = numeric_features + categorical_features


# -----------------------------
# Precision@50
# -----------------------------

def precision_at_50(y_true, scores):
    temp = pd.DataFrame({
        "actual": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    top_50 = temp.sort_values(
        "score",
        ascending=False
    ).head(50)

    return top_50["actual"].mean()


# -----------------------------
# Preprocessing
# -----------------------------

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


# -----------------------------
# Model
# -----------------------------

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])


# ============================================================
# BEFORE: Week-5 style random stratified split
# ============================================================

X = df[feature_columns]
y = df["is_declining_label"]

X_train_before, X_val_before, y_train_before, y_val_before = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_before = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_before.fit(
    X_train_before,
    y_train_before
)

scores_before = model_before.predict_proba(
    X_val_before
)[:, 1]

precision_before = precision_at_50(
    y_val_before,
    scores_before
)


# ============================================================
# AFTER: client-grouped split
# ============================================================

clients = df["client_id"].dropna().unique()

train_clients, val_clients = train_test_split(
    clients,
    test_size=0.20,
    random_state=42
)

train_df = df[
    df["client_id"].isin(train_clients)
].copy()

val_df = df[
    df["client_id"].isin(val_clients)
].copy()

X_train_after = train_df[feature_columns]
y_train_after = train_df["is_declining_label"]

X_val_after = val_df[feature_columns]
y_val_after = val_df["is_declining_label"]

model_after = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model_after.fit(
    X_train_after,
    y_train_after
)

scores_after = model_after.predict_proba(
    X_val_after
)[:, 1]

precision_after = precision_at_50(
    y_val_after,
    scores_after
)


# -----------------------------
# Results
# -----------------------------

comparison = pd.DataFrame({
    "Validation design": [
        "Week-5 random split",
        "Client-grouped split"
    ],
    "Validation rows": [
        len(X_val_before),
        len(X_val_after)
    ],
    "Precision@50": [
        precision_before,
        precision_after
    ]
})

comparison["Precision@50"] = comparison[
    "Precision@50"
].round(3)

print(comparison.to_string(index=False))

print("\nClient overlap in grouped split:")

overlap = len(
    set(train_clients) &
    set(val_clients)
)

print(overlap)

print("\nGrouped split decline rate:")
print(
    round(
        y_val_after.mean() * 100,
        2
    ),
    "%"
)


   Validation design  Validation rows  Precision@50
 Week-5 random split             6000          0.62
Client-grouped split             3419          0.54

Client overlap in grouped split:
0

Grouped split decline rate:
52.38 %


## 3. Leakage audit

## Leakage audit

The final model features exclude `trend_direction` and `trend_pct`, because these fields are used to construct the target and would directly reveal the outcome. `content_id` and `client_id` are also excluded from the model features. I will check the final feature list and measure the relationship between numeric features and the label as an additional warning check.

In [3]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/PrishaSolanki-coder/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

df["is_declining_label"] = (
    df["trend_direction"].astype(str).str.lower() == "down"
).astype(int)


# Final model feature set
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

categorical_features = [
    "content_type",
    "main_intent",
    "provider_used",
    "model_used"
]

final_features = numeric_features + categorical_features


# Columns that must never be model features
banned_columns = [
    "trend_direction",
    "trend_pct",
    "is_declining_label",
    "content_id",
    "client_id"
]


# -----------------------------
# Direct leakage check
# -----------------------------

leakage_columns = [
    col for col in final_features
    if col in banned_columns
]

print("Final feature count:", len(final_features))
print("Banned columns found in features:", leakage_columns)

if len(leakage_columns) == 0:
    print("LEAKAGE CHECK: PASSED")
else:
    print("LEAKAGE CHECK: FAILED")


# -----------------------------
# Correlation warning check
# -----------------------------

correlations = []

for col in numeric_features:
    correlation = df[col].corr(
        df["is_declining_label"]
    )

    correlations.append({
        "feature": col,
        "correlation": correlation
    })

correlation_df = pd.DataFrame(
    correlations
)

correlation_df["absolute_correlation"] = (
    correlation_df["correlation"].abs()
)

correlation_df = correlation_df.sort_values(
    "absolute_correlation",
    ascending=False
)

print("\nTop numeric relationships with label:")
print(
    correlation_df[
        ["feature", "correlation"]
    ].head(10).round(3).to_string(index=False)
)


Final feature count: 14
Banned columns found in features: []
LEAKAGE CHECK: PASSED

Top numeric relationships with label:
               feature  correlation
            word_count        0.090
days_since_last_update        0.081
                   ctr       -0.062
            clicks_90d       -0.040
          avg_position       -0.029
          sessions_90d       -0.023
         search_volume       -0.019
       impressions_90d       -0.018
                   cpc       -0.017
           competition       -0.009


## 4. Claim rewrite
## Claim rewrite

My original claim was:

"Machine learning can accurately identify which pages need to be refreshed."

A safer claim is:

"On this dataset, the Logistic Regression model was able to rank some declining pages near the top of the validation set, as measured by Precision@50. Performance was measured on unseen clients using a client-grouped split, so the result supports the model as a decision-support tool for prioritizing pages for review rather than proving that a page needs a refresh."

In [4]:
print(
    "Week-5 random-split Precision@50:",
    round(precision_before, 3)
)

print(
    "Client-grouped Precision@50:",
    round(precision_after, 3)
)

print(
    "Change:",
    round(
        (precision_after - precision_before) * 100,
        2
    ),
    "percentage points"
)

Week-5 random-split Precision@50: 0.62
Client-grouped Precision@50: 0.54
Change: -8.0 percentage points


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.